# 30 — Judge benchmarks (Kaggle, 2×T4)

Runs jev-lite Stage A, NLI and Laya on four judge benchmarks (`jevlite.data.public.judge_bench`): **ContextualJudgeBench**, **RAGBench**, **HaluBench**, **FaithBench**. Eval only, never trained on.

jev-lite runs twice, to see how it behaves beyond its training length:
- `jevlite_512`: the training length of Stage A; longer inputs are truncated (head + tail of the state);
- `jevlite_8192`: ModernBERT's maximum, no truncation except the longest RAGBench documents (CUAD, TechQA).

NLI and Laya are limited to 512 either way.

- **Settings:** Accelerator → GPU T4 x2, Internet → On. No secrets needed (the model and the datasets are public).
- **GPUs:** `jevlite_8192` (the longest run) on GPU 0; `jevlite_512` → NLI → Laya on GPU 1, one after another.
- **Smoke run first:** run all cells with `LIMIT = 40` (40 records per benchmark, 20 pairs for CJB) and check speed in the logs. Then set `LIMIT = None` and **Save Version → Save & Run All (Commit)**.
- **Hand-off** (on the laptop): last cell.

In [ ]:
# Environment: code from GitHub at a fixed commit, kept outside /kaggle/working so it is not saved as output.
COMMIT = "Judge_bench"   # branch or commit hash; a hash makes the run reproducible

!rm -rf /tmp/JEV_Judge && git clone -q https://github.com/Nefelim584/JEV_Judge.git /tmp/JEV_Judge
!cd /tmp/JEV_Judge && git checkout -q {COMMIT} && git log -1 --format='%h %ad %s' --date=short
!pip install -q --no-deps --ignore-requires-python -e /tmp/JEV_Judge
!pip install -q loguru
!python --version
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Parameters
LIMIT = 40                        # smoke run: records per benchmark (pairs/2 for CJB); None = full benchmarks
MODEL_REPO = "Nefelim584/jev-lite-stage-a"
BENCHMARKS = ["contextual_judge_bench", "ragbench", "halubench", "faithbench"]
REPO = "/tmp/JEV_Judge"
WORK = "/tmp/judge_bench"         # converted data and the model: not saved
OUT = "/kaggle/working"           # notebook output: predictions, logs, report

# One queue per GPU; the runs of a queue go one after another. name: predict_baseline.py arguments
QUEUES = {
    0: {"jevlite_8192": "--baseline jevlite --model {model} --max-len 8192 --fp16"},
    1: {"jevlite_512": "--baseline jevlite --model {model} --max-len 512 --fp16",
        "nli": "--baseline nli --fp16 --block 64",
        "laya": "--baseline laya"},
}

## 1. Model and data

The Hub repo has the `final/` layout (`encoder/`, `heads.pt`, `jevlite.yaml`). The converters are deterministic, so record ids match a local conversion (and `docs/judge_bench/overlap_ids.txt`). Conversion takes a few minutes (~70 MB of downloads).

In [ ]:
import json, os, random, subprocess, sys
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL = snapshot_download(MODEL_REPO, local_dir=f"{WORK}/model")
print("model:", MODEL, sorted(os.listdir(MODEL)))

DATA = Path(WORK) / "data"
subprocess.run([sys.executable, f"{REPO}/scripts/convert_public.py", "--datasets", ",".join(BENCHMARKS),
                "--out", str(DATA), "--raw-dir", f"{WORK}/raw"], check=True, cwd=REPO)

In [ ]:
# Input files: the full benchmarks, or a fixed random sample of each for the smoke run.
def sample(name: str, n: int) -> Path:
    rows = [json.loads(line) for line in (DATA / f"{name}.jsonl").open()]
    rng = random.Random(0)
    if name == "contextual_judge_bench":  # whole pairs, both orders
        groups = set(rng.sample(sorted({r["group"] for r in rows}), n // 2))
        rows = [r for r in rows if r["group"] in groups]
    else:
        rows = rng.sample(rows, min(n, len(rows)))
    path = DATA / "smoke" / f"{name}.jsonl"
    path.parent.mkdir(exist_ok=True)
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))
    return path

FILES = [str(sample(n, LIMIT)) if LIMIT else str(DATA / f"{n}.jsonl") for n in BENCHMARKS]
for f in FILES:
    print(f, sum(1 for _ in open(f)))

## 2. Predictions

Each GPU runs its queue in a background process; logs go to `OUT/logs/<run>.log`. `--resume` keeps finished predictions, so re-running this cell after an interruption continues where it stopped (in the same session). Records a model cannot encode (e.g. a prompt longer than its max_len) go to `<run>.jsonl.skipped.jsonl`.

In [ ]:
PREDS, LOGS = Path(OUT) / "preds", Path(OUT) / "logs"
PREDS.mkdir(exist_ok=True); LOGS.mkdir(exist_ok=True)

def queue_script(runs: dict) -> str:
    lines = ["set -x"]
    for name, args in runs.items():
        a = args.format(model=MODEL)
        lines.append(f"{sys.executable} {REPO}/scripts/predict_baseline.py {a} --data {' '.join(FILES)} "
                     f"--out {PREDS}/{name}.jsonl --resume > {LOGS}/{name}.log 2>&1")
    return "\n".join(lines)

procs = {}
for gpu, runs in QUEUES.items():
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    procs[gpu] = subprocess.Popen(["bash", "-c", queue_script(runs)], env=env, cwd=REPO)
    print(f"GPU {gpu}: {', '.join(runs)} (pid {procs[gpu].pid})")

In [ ]:
# Wait for both queues, printing the last progress line of every log each minute.
import time

def progress(name: str) -> str:
    log = LOGS / f"{name}.log"
    if not log.exists():
        return "waiting"
    text = log.read_text().replace("\r", "\n").strip().splitlines()
    return text[-1][:160] if text else "starting"

while any(p.poll() is None for p in procs.values()):
    print(time.strftime("%H:%M"), " | ".join(f"{n}: {progress(n)}" for runs in QUEUES.values() for n in runs), flush=True)
    time.sleep(60)
for gpu, p in procs.items():
    print(f"GPU {gpu}: exit code {p.returncode}")
for runs in QUEUES.values():
    for n in runs:
        print(f"== {n}:", progress(n))

## 3. Report

On the records every model predicted, minus the 49 whose passages appear in the Stage A training data. `fits512` = records that fit 512 tokens untruncated.

In [ ]:
names = [n for runs in QUEUES.values() for n in runs if (PREDS / f"{n}.jsonl").exists()]
cmd = (f"{sys.executable} {REPO}/scripts/judge_bench_report.py --data {' '.join(FILES)} "
       f"--pred {' '.join(f'{n}={PREDS}/{n}.jsonl' for n in names)} "
       f"--exclude {REPO}/docs/judge_bench/overlap_ids.txt --tokenizer {MODEL}/encoder "
       f"--out {OUT}/report --bootstrap {0 if LIMIT else 1000}")
print(cmd)
!{cmd}

## Hand-off (on the laptop, from the repo root)

```bash
kaggle kernels output <user>/<notebook-slug> -p data/judge_bench/kaggle   # preds/, logs/, report/
```

The report can be rebuilt locally with `scripts/judge_bench_report.py` on `data/public/<benchmark>.jsonl` and the downloaded predictions.